# TRUST-MRI pipeline by hand

This notebook runs the pipeline step by step on **one patient**:

1. load the images to analyse;
2. preprocess (crop to the brain, intensities in [0, 1]);
3. **A.** compare the imputed images with the real image;
4. **B.** build local patch maps;
5. **C.** measure the spread of the N imputed images;
6. **D.** evaluate the segmentation;
7. compute uncertainty and the ANOVA (imputation vs prompt);
8. summarise.

Steps 3, 4 and 6 need the ground truth; they are skipped otherwise.
Run the notebook from the `trust_pipeline/` folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trust_mri_eval.config import Config
from trust_mri_eval.data.io import load_patient
from trust_mri_eval.data.synthetic import generate_patient
from trust_mri_eval.preprocessing import prepare
from trust_mri_eval.patches import PatchGrid
from trust_mri_eval.metrics.ssim import ssim_map
from trust_mri_eval.metrics.image_global import image_global_metrics
from trust_mri_eval.metrics.image_local import voxel_error_maps, patch_maps
from trust_mri_eval.metrics.distribution import mean_variance_maps, pairwise_ssim, pairwise_psnr, interval_coverage
from trust_mri_eval.metrics.segmentation import segmentation_metrics
from trust_mri_eval.uncertainty.confidence_maps import agreement_map, entropy_map
from trust_mri_eval.uncertainty.anova import anova_maps, anova_scalar

## 1. Load the images

Two options:

- **`"folder"`**: a patient folder containing NIfTI files with these names:
  - `imputed_n0.nii.gz` … `imputed_n{N-1}.nii.gz`: the N imputed images;
  - `seg_ET_n{n}_k{k}.nii.gz`: tumour probability for image n and prompt k;
  - `brain_mask.nii.gz` (optional);
  - `gt_image.nii.gz` and `gt_mask.nii.gz` (optional, ground truth);
- **`"synthetic"`**: a generated fake patient, to try the notebook.

In [ ]:
SOURCE = "synthetic"                        # "synthetic" or "folder"
PATIENT_FOLDER = "path/to/patient"          # used only if SOURCE == "folder"
REGION = "ET"

if SOURCE == "folder":
    patient = load_patient(PATIENT_FOLDER, region=REGION)
else:
    patient = generate_patient(0, Config(volume_shape=(96, 96, 64), syn_tumour_radius=(4, 12)))

print("patient        :", patient.patient_id)
print("imputed images :", patient.imputed.shape, "  (N, X, Y, Z)")
print("segmentations  :", patient.seg_probs.shape, "  (N, K, X, Y, Z)")
print("ground truth   :", patient.has_ground_truth)

## 2. Preprocessing

`prepare` crops every volume to the brain bounding box, rescales each image to [0, 1]
(0.5th and 99.5th percentiles inside the brain), binarises the segmentations (threshold 0.5) and
computes the **consensus** (mean of the N × K probabilities, thresholded at 0.5).

In [ ]:
prep = prepare(patient)
brain = prep.brain
N, K = prep.probs.shape[:2]

# axial slice where the tumour is largest (reference if available, else consensus)
tumour_mask = prep.gt_mask if prep.has_ground_truth else prep.consensus
z = int(np.argmax(tumour_mask.sum(axis=(0, 1))))


def show(images, cmap="gray", **options):
    """Show slice z of each volume, side by side (options: vmin, vmax...)."""
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    for ax, (name, volume) in zip(np.atleast_1d(axes), images.items()):
        im = ax.imshow(volume[:, :, z].T, origin="lower", cmap=cmap, **options)
        ax.set_title(name)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()


images = {"imputed (n = 0)": prep.imputed[0]}
if prep.has_ground_truth:
    images["real"] = prep.gt_image
show(images)
show({"segmentation (n = 0, k = 0)": prep.probs[0, 0], "consensus": prep.consensus.astype(float)}, cmap="Blues")

## 3. A. Imputed image vs real image

For each imputed image, inside the brain mask: L1 and L2 errors, PSNR, and SSIM over the whole
brain, the tumour and the healthy brain.

In [ ]:
table_A = pd.DataFrame()
if prep.has_ground_truth:
    A = image_global_metrics(prep.gt_image, prep.imputed, brain, prep.tumour)
    table_A = pd.DataFrame(A, index=[f"image {n}" for n in range(N)])
    table_A.loc["mean"] = table_A.iloc[:N].mean()
    table_A.loc["std"] = table_A.iloc[:N].std(ddof=0)
else:
    print("No ground truth: step skipped.")
table_A.round(4)

## 4. B. Local patch maps

Each map is averaged over patches of 3×3×3 voxels (change `PATCH_SIZE`). The **failure map** marks
the patches whose L1 error exceeds the 90th percentile. In the full pipeline this threshold is learned
on the training patients; here it is computed on this patient alone.

In [ ]:
PATCH_SIZE = (3, 3, 3)            # (8, 8, 8) for a coarser map

if prep.has_ground_truth:
    grid = PatchGrid.over(prep.shape, PATCH_SIZE)
    errors = voxel_error_maps(prep.gt_image, prep.imputed)
    local_ssim = np.mean([ssim_map(prep.gt_image, image) for image in prep.imputed], axis=0)
    _, variance = mean_variance_maps(prep.imputed)
    patches = patch_maps({"local SSIM": local_ssim, "L1 error": errors["l1"], "L2 error": errors["l2"],
                          "variance": variance}, grid, brain)

    threshold = np.nanpercentile(patches["L1 error"], 90)
    patches["failure"] = (patches["L1 error"] > threshold).astype(float)
    print(f"L1 threshold = {threshold:.4f}, failing patches: {np.nanmean(patches['failure'][~np.isnan(patches['L1 error'])]):.1%}")
    show({name: grid.expand(values) for name, values in patches.items()}, cmap="Blues")
else:
    print("No ground truth: step skipped.")

## 5. C. Spread of the N imputed images

No ground truth is needed, except for the **coverage**: the fraction of voxels where the true
intensity falls between the 5th and 95th percentiles of the N images.

In [ ]:
mean_image, variance = mean_variance_maps(prep.imputed)
pair_ssim, pair_ssim_map = pairwise_ssim(prep.imputed, brain)
pair_psnr = pairwise_psnr(prep.imputed, brain)

C = {
    "mean variance (brain)": variance[brain].mean(),
    "mean variance (tumour)": variance[tumour_mask].mean(),
    "pairwise SSIM (mean)": np.mean(pair_ssim),
    "pairwise SSIM (std)": np.std(pair_ssim),
    "pairwise PSNR (mean, dB)": np.mean(pair_psnr),
}
if prep.has_ground_truth:
    C["coverage of the [5, 95] % interval"] = interval_coverage(prep.imputed, prep.gt_image, brain)

show({"variance across images": variance, "pairwise SSIM": pair_ssim_map}, cmap="Blues")
pd.Series(C).round(4)

## 6. D. Segmentation

The consensus is compared with the reference mask: Dice, NSD (1 mm tolerance), HD95, BraTS 2023
lesion-wise Dice, and lesion-level precision / recall / F1.

In [ ]:
D = {}
if prep.has_ground_truth:
    D = segmentation_metrics(prep.consensus, prep.gt_mask, prep.spacing, nsd_tolerance_mm=1.0)
else:
    print("No ground truth: step skipped.")
pd.Series(D, dtype=float).round(4)

## 7. Uncertainty and ANOVA (no ground truth)

- **Agreement**: fraction of the N × K masks that see tumour at each voxel.
- **Entropy** of the mean probability (0 = certain, 1 bit = 50/50).
- **ANOVA**: at each voxel of the region of interest, the variance of the probabilities is split
  into a part due to the imputation (n), a part due to the prompt (k) and an interaction part.

In [ ]:
agreement = agreement_map(prep.masks)
entropy = entropy_map(prep.mean_prob)
anova = anova_maps(prep.probs, prep.roi)          # computed only inside the region of interest

total_variance = anova["var_total"][prep.roi].sum()
shares = {f"{c} share": anova[f"var_{c}"][prep.roi].sum() / total_variance if total_variance > 0 else np.nan
          for c in ("imputation", "prompt", "interaction")}
print(pd.Series(shares).round(3))

show({"agreement": agreement, "entropy": entropy}, cmap="Blues", vmin=0, vmax=1)
show({f"{c} fraction": anova[f"frac_{c}"] for c in ("imputation", "prompt", "interaction")},
     cmap="Blues", vmin=0, vmax=1)

The same decomposition applies to the **predicted tumour volume** of each pair
(image n, prompt k).

In [ ]:
volumes = prep.masks.sum(axis=(2, 3, 4)) * np.prod(prep.spacing)          # (N, K), in mm³
print(pd.DataFrame(volumes, index=[f"image {n}" for n in range(N)], columns=[f"prompt {k}" for k in range(K)]))
volume_anova = anova_scalar(volumes)
pd.Series({c: volume_anova[f"frac_{c}"] for c in ("imputation", "prompt", "interaction")}, name="fraction").round(3)

## 8. Summary

All metrics in one table, saved as CSV next to the notebook.

The **confidence model** (the next step of the pipeline) learns from several patients; run it from
the command line:
`python -m trust_mri_eval.cli --data-dir <patients folder>`.

In [ ]:
summary = pd.concat({
    "A (mean over the N images)": table_A.loc["mean"] if "mean" in table_A.index else pd.Series(dtype=float),
    "C": pd.Series(C),
    "D": pd.Series(D, dtype=float),
    "ANOVA": pd.Series(shares),
})
summary.to_csv(f"metrics_{patient.patient_id}.csv", header=["value"])
summary.round(4).to_frame("value")